# S2b: B0 (base Tiny Aya En-Thinker, `/no_think`) on one dev episode

**Before running:**
1. Settings: Accelerator = GPU T4 x1 (or P100), Internet = On.
2. Add-ons > Secrets: `HF_TOKEN` and `API_DEEPSEEK` (attach both to this notebook).
3. Add a **private** dataset containing `/tmp/agentic_distill_src.zip` (made on the Mac by `scripts/make_kaggle_bundle.sh`), named e.g. `agentic-distill-src`, and attach it.

Stops (does not work around) if Python < 3.12, no GPU, or an install fails.

In [ ]:
import sys, subprocess, os, glob
print(sys.version)
assert sys.version_info >= (3, 12), "tau2 needs Python >= 3.12; stop and report (do not hack around)"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
zips = glob.glob("/kaggle/input/*/agentic_distill_src.zip")
print("bundle:", zips)
assert zips, "attach the private bundle dataset"


In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["API_DEEPSEEK"] = s.get_secret("API_DEEPSEEK")
print("secrets loaded:", bool(os.environ["HF_TOKEN"]), bool(os.environ["API_DEEPSEEK"]))   # values never printed


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
cd /kaggle/working/repo && unzip -q /kaggle/input/*/agentic_distill_src.zip
mkdir -p third_party && git clone -q https://github.com/sierra-research/tau2-bench third_party/tau2-bench
git -C third_party/tau2-bench checkout -q c06b07dd7c4d4ec61ba6ff9bcdf0b15dd7ef36d4
pip install -q -e third_party/tau2-bench audioop-lts pyyaml transformers accelerate
python -c "import tau2, transformers, torch; print('tau2 ok; torch', torch.__version__, torch.cuda.is_available())"


In [ ]:
%%bash
cd /kaggle/working/repo
PYTHONPATH=src python -m pytest -q tests/test_formatter.py -x 2>&1 | tail -3   # skips if tokenizer dir absent; fine


In [ ]:
%%bash
cd /kaggle/working/repo
# Task 75 (single exchange). Add --allow-peak only if you accept 2x DeepSeek user-sim prices.
PYTHONPATH=src python -m agentic_distill.student_run --tasks 75 --trials 1 --slice s3_pilot --label b0 \
  --out-root /kaggle/working/out 2>&1 | grep -v DEBUG | tail -25


In [ ]:
import json, glob
for p in glob.glob("/kaggle/working/out/data/raw/*/episodes.jsonl"):
    for l in open(p): print(json.dumps(json.loads(l), indent=1))
# Download /kaggle/working/out/data/raw/<run>/ (traces + episodes.jsonl) back to the Mac for review.
